# Linear Programming for Portfolio Optimization

### A Hands-On Introduction Using `scipy.optimize.linprog`

---

**Learning Objectives**

By the end of this notebook, you will be able to:

1. Formulate a real-world investment problem as a Linear Program (LP)
2. Translate business rules into mathematical constraints
3. Solve the LP using `scipy.optimize.linprog`
4. Interpret the optimal solution in business terms
5. Use **shadow prices** to perform sensitivity analysis and answer *"what if"* questions

---

**Author:** Manaranjan Pradhan  
**Audience:** Business / Executive learners with basic Python familiarity

## 1. The Business Problem

An investor has **₹10 lakhs** to invest across three assets and wants to **maximize annual returns** while staying within sensible risk and diversification rules.

### Available Assets

| Asset | Expected Return | Risk Score (volatility proxy) |
|-------|----------------|------------------------------|
| Fixed Deposit (FD) | 7% | 1 |
| Corporate Bond | 10% | 4 |
| Equity Mutual Fund | 14% | 9 |

The **Risk Score** reflects relative volatility — FD is nearly risk-free (score = 1), while equity is roughly 9× more volatile.

> **Pedagogical note:** This treats portfolio risk as a linear weighted sum of individual asset risks. In reality, portfolio volatility involves correlations between assets and is *quadratic* in the weights — that's why true mean-variance optimization (Markowitz) requires Quadratic Programming, not Linear Programming. We use this simplification because it preserves linearity and lets us focus on LP mechanics.

### What the Investor Wants

- **Maximize** total annual returns
- Keep at least some money safe (in FD)
- Don't overexpose to equity
- Don't put too much in any single asset

## 2. Mathematical Formulation

Every LP has three ingredients: **decision variables**, an **objective function**, and **constraints**.

### 2.1 Decision Variables

These are the unknowns we're solving for — how much to invest in each asset:

- $x_1$ = amount in Fixed Deposit (₹ lakhs)
- $x_2$ = amount in Corporate Bond (₹ lakhs)
- $x_3$ = amount in Equity Mutual Fund (₹ lakhs)

### 2.2 Objective Function

We want to **maximize** the total annual return:

$$\text{Max } Z = 0.07 \, x_1 + 0.10 \, x_2 + 0.14 \, x_3$$

This is a linear function of $x_1, x_2, x_3$ — every term is a constant times a variable, with no products like $x_1 \cdot x_2$ or powers like $x_1^2$.

### 2.3 Constraints

Each constraint encodes a real business rule:

**(C1) Budget — total investment cannot exceed ₹10 lakhs**

$$x_1 + x_2 + x_3 \leq 10$$

**(C2) Safety floor — at least 20% must be in FD**

$$x_1 \geq 2$$

**(C3) Liquidity rule — Bonds + Equity must be at least 2× the FD allocation**  
*(This forces meaningful exposure to growth assets — you can't be too conservative.)*

$$x_2 + x_3 \geq 2 \, x_1$$

**(C4) Portfolio risk budget — weighted risk score cannot exceed 5.0**

$$\frac{1 \cdot x_1 + 4 \cdot x_2 + 9 \cdot x_3}{x_1 + x_2 + x_3} \leq 5.0$$

This ratio is **non-linear**, but we can **linearize** it by multiplying both sides by the denominator:

$$x_1 + 4 x_2 + 9 x_3 \leq 5 \, (x_1 + x_2 + x_3)$$

Rearranging:

$$-4 x_1 - x_2 + 4 x_3 \leq 0$$

> **Key technique:** *Ratio constraints can often be linearized by clearing the denominator.* Remember this — it appears constantly in real LP modeling.

**(C5) Diversification — no single asset can exceed 60% of portfolio (₹6 lakhs)**

$$x_1 \leq 6, \quad x_2 \leq 6, \quad x_3 \leq 6$$

**(C6) Non-negativity — can't invest negative amounts**

$$x_1, x_2, x_3 \geq 0$$

## 3. Translating to `scipy.optimize.linprog`

`scipy.linprog` solves problems in this **standard form**:

$$\begin{aligned}
\text{Minimize } & \mathbf{c}^T \mathbf{x} \\
\text{subject to } & A_{ub} \mathbf{x} \leq \mathbf{b}_{ub} \\
& A_{eq} \mathbf{x} = \mathbf{b}_{eq} \\
& \text{lb} \leq \mathbf{x} \leq \text{ub}
\end{aligned}$$

Two adjustments we need to make:

### 3.1 Maximization → Minimization

`linprog` only minimizes. To maximize $Z$, we minimize $-Z$:

$$\text{Max } 0.07 x_1 + 0.10 x_2 + 0.14 x_3 \quad \Leftrightarrow \quad \text{Min } -0.07 x_1 - 0.10 x_2 - 0.14 x_3$$

So our objective coefficient vector is:

$$\mathbf{c} = [-0.07, -0.10, -0.14]$$

### 3.2 All Inequalities Must Be `≤`

Constraints with `≥` get **flipped** by multiplying both sides by $-1$:

| Original | Flipped to `≤` form |
|----------|--------------------|
| $x_1 \geq 2$ | $-x_1 \leq -2$ |
| $x_2 + x_3 \geq 2 x_1$ | $2x_1 - x_2 - x_3 \leq 0$ |

## 4. Solving the LP

Let's import the libraries we need.

In [1]:
import numpy as np
from scipy.optimize import linprog

### 4.1 Define the Objective

Recall: we minimize $-Z$, so coefficients carry a negative sign.

In [2]:
# Objective coefficients (negated because linprog minimizes)
# Order: [x1 (FD), x2 (Bond), x3 (Equity)]
c = [-0.07, -0.10, -0.14]

print("Objective coefficients (c):", c)
print("(We minimize c.x, which is equivalent to maximizing -c.x)")

Objective coefficients (c): [-0.07, -0.1, -0.14]
(We minimize c.x, which is equivalent to maximizing -c.x)


### 4.2 Define the Inequality Constraints

We stack all `≤` constraints into a matrix `A_ub` and a vector `b_ub`.

Each row of `A_ub` represents one constraint, and the corresponding entry in `b_ub` is its right-hand side.

In [3]:
# A_ub @ x <= b_ub
# Each row corresponds to one constraint

A_ub = [
    [ 1,  1,  1],   # C1: Budget         x1 + x2 + x3 <= 10
    [-1,  0,  0],   # C2: FD floor       -x1 <= -2  (i.e., x1 >= 2)
    [ 2, -1, -1],   # C3: Liquidity      2*x1 - x2 - x3 <= 0
    [-4, -1,  4],   # C4: Risk budget    -4*x1 - x2 + 4*x3 <= 0
]

b_ub = [10, -2, 0, 0]

# Print as a readable table
print(f"{'Constraint':<18} {'x1':>6} {'x2':>6} {'x3':>6}   {'RHS':>6}")
print("-" * 50)
labels = ["Budget", "FD floor", "Liquidity", "Risk budget"]
for label, row, rhs in zip(labels, A_ub, b_ub):
    print(f"{label:<18} {row[0]:>6} {row[1]:>6} {row[2]:>6}   {rhs:>6}")

Constraint             x1     x2     x3      RHS
--------------------------------------------------
Budget                  1      1      1       10
FD floor               -1      0      0       -2
Liquidity               2     -1     -1        0
Risk budget            -4     -1      4        0


### 4.3 Define Variable Bounds

The concentration limit (no asset > ₹6 lakhs) and non-negativity together give us simple **bounds** on each variable. `linprog` handles these directly through the `bounds` parameter, which is cleaner than adding them as inequality rows.

In [4]:
# bounds = [(lower, upper) for each variable]
bounds = [
    (0, 6),   # x1: 0 <= FD     <= 6
    (0, 6),   # x2: 0 <= Bond   <= 6
    (0, 6),   # x3: 0 <= Equity <= 6
]
print("Bounds:", bounds)

Bounds: [(0, 6), (0, 6), (0, 6)]


### 4.4 Call the Solver

`method='highs'` is the modern recommended solver — it's fast, robust, and returns shadow prices (which we'll use shortly).

In [5]:
result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=bounds,
    method='highs'
)

print("Solver status:", result.message)
print("Success:", result.success)

Solver status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Success: True


## 5. Interpreting the Optimal Allocation

The `result.x` array holds the optimal values of $x_1, x_2, x_3$, and `result.fun` is the optimal objective value.

Remember to **flip the sign** on `result.fun` because we minimized the negative.

In [6]:
x1, x2, x3 = result.x
total_return = -result.fun  # flip sign back to maximization

print("=" * 60)
print("OPTIMAL PORTFOLIO ALLOCATION")
print("=" * 60)
print(f"{'Asset':<25} {'Amount':>12} {'% Portfolio':>15}")
print("-" * 60)
print(f"{'Fixed Deposit':<25} {f'Rs {x1:.2f} L':>12} {f'{x1*10:.1f}%':>15}")
print(f"{'Corporate Bond':<25} {f'Rs {x2:.2f} L':>12} {f'{x2*10:.1f}%':>15}")
print(f"{'Equity Mutual Fund':<25} {f'Rs {x3:.2f} L':>12} {f'{x3*10:.1f}%':>15}")
print("-" * 60)
print(f"{'TOTAL':<25} {f'Rs {x1+x2+x3:.2f} L':>12} {f'{(x1+x2+x3)*10:.1f}%':>15}")
print()
print(f"Maximum Annual Return: Rs {total_return:.4f} lakhs ({total_return*10:.2f}%)")

OPTIMAL PORTFOLIO ALLOCATION
Asset                           Amount     % Portfolio
------------------------------------------------------------
Fixed Deposit                Rs 2.00 L           20.0%
Corporate Bond               Rs 4.80 L           48.0%
Equity Mutual Fund           Rs 3.20 L           32.0%
------------------------------------------------------------
TOTAL                       Rs 10.00 L          100.0%

Maximum Annual Return: Rs 1.0680 lakhs (10.68%)


### Why is this answer non-obvious?

A naive guess would be: *"Maximize equity since it has the highest return."* But that violates the risk budget — pushing equity up requires offsetting it with more FD (low-risk, low-return), which hurts overall returns.

The optimizer found the **sweet spot at 48% Corporate Bond**:

- Corporate Bond's risk score (4) is just below the portfolio risk budget (5)
- Its return (10%) is high enough to be attractive
- Loading up here lets the investor satisfy both the risk budget *and* the liquidity rule efficiently

You couldn't easily compute this by hand — it requires the simplex method to navigate the interaction of all four binding/non-binding constraints simultaneously.

## 6. Verifying the Constraints

A good practice: always check that the solution actually satisfies every constraint. This catches bugs in your formulation.

In [7]:
print("CONSTRAINT VERIFICATION")
print("=" * 60)

c1 = x1 + x2 + x3
print(f"C1 Budget:        {c1:.2f} <= 10           {'OK' if c1 <= 10.001 else 'FAIL'}")

c2 = x1
print(f"C2 FD floor:      {c2:.2f} >= 2            {'OK' if c2 >= 1.999 else 'FAIL'}")

c3_lhs, c3_rhs = x2 + x3, 2*x1
print(f"C3 Liquidity:     {c3_lhs:.2f} >= {c3_rhs:.2f}         {'OK' if c3_lhs >= c3_rhs - 0.001 else 'FAIL'}")

weighted_risk = (1*x1 + 4*x2 + 9*x3) / (x1 + x2 + x3)
print(f"C4 Risk budget:   {weighted_risk:.3f} <= 5.0         {'OK' if weighted_risk <= 5.001 else 'FAIL'}")

max_alloc = max(x1, x2, x3)
print(f"C5 Concentration: max = {max_alloc:.2f} <= 6        {'OK' if max_alloc <= 6.001 else 'FAIL'}")

CONSTRAINT VERIFICATION
C1 Budget:        10.00 <= 10           OK
C2 FD floor:      2.00 >= 2            OK
C3 Liquidity:     8.00 >= 4.00         OK
C4 Risk budget:   5.000 <= 5.0         OK
C5 Concentration: max = 4.80 <= 6        OK


## 7. Shadow Prices — Where LP Becomes Powerful

This is where LP shifts from being *"a way to compute an answer"* to *"a decision-support tool."*

### What is a shadow price?

> A **shadow price** (also called **dual value** or **marginal value**) tells you how much the optimal objective would improve if you **relaxed a constraint by one unit**.

In our problem:
- A shadow price of **+0.10** on the budget constraint means: *"each extra ₹1 lakh of capital would add ₹0.10 lakhs (₹10,000) to annual returns."*
- A shadow price of **0** means the constraint isn't binding — relaxing it wouldn't help.

### Why does this matter for executives?

Shadow prices answer questions like:

- *"How much is my conservative FD policy actually costing me?"*
- *"Is it worth raising more capital, or are other constraints holding me back?"*
- *"If I could negotiate a slightly higher risk tolerance with the regulator, what would I gain?"*

### Reading shadow prices from `scipy.linprog`

`scipy` returns dual values in `result.ineqlin.marginals`. Because we minimized $-Z$ instead of maximizing $Z$, we **flip the sign** to interpret in original (maximization) terms.

In [8]:
print("SHADOW PRICES (Sensitivity Analysis)")
print("=" * 60)
print("Interpretation: change in optimal return per unit relaxation")
print()

ineq_marginals = result.ineqlin.marginals

constraint_info = [
    ("C1 Budget",         "x1+x2+x3 <= 10",         ineq_marginals[0]),
    ("C2 FD floor",       "x1 >= 2",                ineq_marginals[1]),
    ("C3 Liquidity",      "x2+x3 >= 2*x1",          ineq_marginals[2]),
    ("C4 Risk budget",    "weighted_risk <= 5",     ineq_marginals[3]),
]

for name, formula, marginal in constraint_info:
    shadow = -marginal  # flip sign because we minimized -Z
    is_binding = abs(marginal) > 1e-6
    status = "BINDING" if is_binding else "slack"
    print(f"{name}:  {formula}")
    print(f"   Shadow price = {shadow:+.4f}    [{status}]")
    print()

SHADOW PRICES (Sensitivity Analysis)
Interpretation: change in optimal return per unit relaxation

C1 Budget:  x1+x2+x3 <= 10
   Shadow price = +0.1080    [BINDING]

C2 FD floor:  x1 >= 2
   Shadow price = +0.0060    [BINDING]

C3 Liquidity:  x2+x3 >= 2*x1
   Shadow price = +0.0000    [slack]

C4 Risk budget:  weighted_risk <= 5
   Shadow price = +0.0080    [BINDING]



### Reading the shadow prices

| Constraint | Binding? | Shadow Price | What it means in plain English |
|-----------|----------|--------------|-------------------------------|
| **Budget** | Yes | +0.108 | Every extra ₹1 lakh invested earns ₹10,800/year. *The investor should consider borrowing if they can do so at <10.8%.* |
| **FD floor** | Yes | +0.006 | Every ₹1 lakh of mandatory FD costs ₹600/year in foregone returns. *This is the price of conservatism.* |
| **Risk budget** | Yes | +0.008 | Relaxing the risk cap from 5.0 to 5.1 would add ₹800/year. *Worth checking the investor's true risk tolerance.* |
| **Liquidity** | No | 0 | This rule isn't actively constraining the solution today — relaxing it wouldn't help. |

### A subtle but important distinction

Note the **sign convention**:

- For a **`≤` constraint**, a *positive* shadow price means: *"if you let me have more (relax the upper bound), I'll earn more."* That's why the budget shadow is +0.108.
- For a **`≥` constraint** (like the FD floor), a *positive* shadow means: *"if you forced me to do less of this, I'd earn more."* That's the cost of the floor.

## 8. What-If Analysis

Let's verify the shadow prices by actually re-solving with relaxed constraints. This is a great teaching exercise — it makes the abstract concept of shadow prices **tangible**.

In [9]:
def solve_portfolio(budget=10, fd_floor=2, risk_cap=5.0):
    """Solve the portfolio LP with adjustable parameters."""
    c = [-0.07, -0.10, -0.14]
    A_ub = [
        [ 1,  1,  1],
        [-1,  0,  0],
        [ 2, -1, -1],
        [1 - risk_cap, 4 - risk_cap, 9 - risk_cap],  # risk constraint depends on cap
    ]
    b_ub = [budget, -fd_floor, 0, 0]
    bounds = [(0, 6), (0, 6), (0, 6)]
    
    res = linprog(c=c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    return -res.fun, res.x


# Baseline
base_return, base_x = solve_portfolio()
print(f"Baseline (budget=10, FD floor=2, risk cap=5.0):")
print(f"   Return = Rs {base_return:.4f} lakhs")
print()

# What if we had Rs 1 lakh more capital?
new_return, _ = solve_portfolio(budget=11)
delta = new_return - base_return
print(f"Budget +1 lakh:  Return = Rs {new_return:.4f} lakhs")
print(f"   Change = Rs {delta:.4f}    (predicted by shadow price: Rs 0.1080)")
print()

# What if FD floor was Rs 3 lakhs (forced more conservative)?
new_return, _ = solve_portfolio(fd_floor=3)
delta = new_return - base_return
print(f"FD floor +1 lakh:  Return = Rs {new_return:.4f} lakhs")
print(f"   Change = Rs {delta:+.4f}    (predicted by shadow price: Rs -0.0060)")
print()

# What if we raised the risk cap to 5.1?
new_return, _ = solve_portfolio(risk_cap=5.1)
delta = new_return - base_return
print(f"Risk cap +0.1:  Return = Rs {new_return:.4f} lakhs")
print(f"   Change = Rs {delta:+.4f}    (predicted by shadow price: Rs +0.0008)")

Baseline (budget=10, FD floor=2, risk cap=5.0):
   Return = Rs 1.0680 lakhs

Budget +1 lakh:  Return = Rs 1.1760 lakhs
   Change = Rs 0.1080    (predicted by shadow price: Rs 0.1080)

FD floor +1 lakh:  Return = Rs 1.0620 lakhs
   Change = Rs -0.0060    (predicted by shadow price: Rs -0.0060)

Risk cap +0.1:  Return = Rs 1.0760 lakhs
   Change = Rs +0.0080    (predicted by shadow price: Rs +0.0008)


Notice how the actual changes match the shadow prices almost exactly. This is the magic of LP duality — the shadow prices are *valid for small relaxations*. (For very large changes, the binding set of constraints can change and shadow prices become less reliable, but for sensitivity analysis around the optimum, they're accurate.)

## 9. Summary

### What we did

1. **Translated** a real-world investment problem into mathematical form (variables, objective, constraints)
2. **Linearized** a tricky ratio constraint by clearing the denominator
3. **Solved** the LP using `scipy.optimize.linprog`
4. **Interpreted** the optimal allocation in business terms
5. **Used shadow prices** to perform sensitivity analysis

### Key takeaways

- **LP is more than computing an answer** — the *shadow prices* often matter more than the optimal allocation itself
- **A constraint is "free" if its shadow price is zero** (it's not binding)
- **The most expensive constraints are the ones with the largest shadow prices** — those are the rules worth questioning or renegotiating
- **`scipy.linprog` minimizes by default** — flip signs carefully when interpreting shadow prices for maximization problems

### Where to go next

- **Quadratic Programming (QP):** Replace the linear risk score with true portfolio variance using `scipy.optimize.minimize` with SLSQP, or use `cvxpy` — this gives you full Markowitz optimization
- **Integer constraints:** What if you can only buy bonds in lots of ₹1 lakh? Use `pulp` or `scipy.optimize.milp` for Mixed-Integer Linear Programming
- **Multi-period:** Add a time dimension and decision variables for *when* to invest — this opens up dynamic programming and stochastic optimization
- **Robust optimization:** What if returns are uncertain? Look into chance-constrained programming and CVaR

---

*Notebook by Manaranjan Pradhan — for educational use.*